# Week 2 — IP102 Dataset Exploration and Integrity Checks

**Project:** Targeted Training Data Cleaning for Pest Classification on IP102  
**Status:** Reconstructed, reproducible notebook based on documented Week 2 procedures. **Not a saved copy of the original Colab session.**

This notebook verifies dataset split sizes, training-class imbalance, minority-class membership, and image integrity. It does not train a model or modify dataset splits.

## 1. Setup

Run on Google Colab or another Python environment. If the dataset is already extracted, skip the optional download cell.

In [ ]:
# Colab only: install if needed
%pip -q install pandas Pillow matplotlib gdown

In [ ]:
from pathlib import Path
from collections import Counter
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_DIR = Path('/content/IP102_extracted/ip102_v1.1')
IMAGE_DIR = DATA_DIR / 'images'
OUTPUT_DIR = Path('/content/ip102_week02_results')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Dataset directory:', DATA_DIR)
print('Exists:', DATA_DIR.exists())

## 2. Optional dataset preparation

If the IP102 dataset is not yet extracted in this runtime, execute the next cell. The public Google Drive folder and archive names reflect the source used in the project; availability may change. The download is approximately several GB.

In [ ]:
# OPTIONAL — run only if DATA_DIR does not exist.
# Uncomment the following lines in Colab if needed:
# !mkdir -p /content/IP102_dataset /content/IP102_extracted
# !gdown --folder "https://drive.google.com/drive/folders/1ntmFAYessXYRLl8yuD_sPqdyPRtzxwyz" -O /content/IP102_dataset
# !tar -xf /content/IP102_dataset/ip102_v1.1.tar -C /content/IP102_extracted

assert DATA_DIR.exists(), 'Dataset not found. Prepare/extract the IP102 dataset first.'
assert IMAGE_DIR.is_dir(), f'Missing images directory: {IMAGE_DIR}' 

## 3. Read the original dataset splits

Each split file is expected to contain a filename and a zero-based class ID per row.

In [ ]:
def read_split(name):
    path = DATA_DIR / f'{name}.txt'
    if not path.is_file():
        raise FileNotFoundError(path)
    df = pd.read_csv(path, sep=r'\s+', header=None, names=['filename', 'class_id'])
    df['class_id'] = df['class_id'].astype(int)
    return df

splits = {name: read_split(name) for name in ('train', 'val', 'test')}
summary = pd.DataFrame([{'split': name, 'images': len(df), 'classes_present': df['class_id'].nunique()}
                        for name, df in splits.items()])
print(summary.to_string(index=False))
print('Total:', sum(len(df) for df in splits.values()))
summary.to_csv(OUTPUT_DIR / 'split_summary.csv', index=False)

## 4. Validate class IDs and split membership

In [ ]:
NUM_CLASSES = 102
for name, df in splits.items():
    assert df['class_id'].between(0, NUM_CLASSES - 1).all(), f'Unexpected class ID in {name}'
    assert not df['filename'].duplicated().any(), f'Duplicate filenames within {name}'

sets = {name: set(df['filename']) for name, df in splits.items()}
for a, b in [('train','val'), ('train','test'), ('val','test')]:
    overlap = sets[a] & sets[b]
    print(f'{a}/{b} filename overlap:', len(overlap))
    assert not overlap, f'Unexpected filename overlap between {a} and {b}'
print('Split checks passed.')

## 5. Training class distribution

In [ ]:
train_counts = splits['train']['class_id'].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index()
counts_df = pd.DataFrame({'class_id': train_counts.index, 'train_count': train_counts.values})
counts_df.to_csv(OUTPUT_DIR / 'train_class_counts.csv', index=False)
print('Largest class:', int(train_counts.idxmax()), '—', int(train_counts.max()), 'images')
print('Smallest class:', int(train_counts.idxmin()), '—', int(train_counts.min()), 'images')
print('Imbalance ratio:', round(train_counts.max() / train_counts.min(), 2))

plt.figure(figsize=(15, 4))
plt.bar(train_counts.index, train_counts.values)
plt.xlabel('Class ID')
plt.ylabel('Training image count')
plt.title('IP102 training class distribution')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'train_class_distribution.png', dpi=150)
plt.show()

## 6. Define the 21 minority (tail) classes

The 21 least-represented training classes are selected once from the original training split. This definition remains fixed in all later experiments.

In [ ]:
TAIL_COUNT = 21
tail_counts = train_counts.sort_values(kind='stable').head(TAIL_COUNT)
TAIL_CLASSES = [int(i) for i in tail_counts.index]
print('Tail class IDs:', TAIL_CLASSES)
print('Tail class training images:', int(tail_counts.sum()))
print('Tail share of training images:', f'{100 * tail_counts.sum() / len(splits["train"]):.2f}%')
tail_df = pd.DataFrame({'class_id': tail_counts.index, 'train_count': tail_counts.values})
tail_df.to_csv(OUTPUT_DIR / 'tail_classes.csv', index=False)
display(tail_df.reset_index(drop=True))

## 7. Check missing and corrupted images

This performs a full image verification pass over all split entries. It may take several minutes. `Image.verify()` checks file integrity without fully decoding pixel data.

In [ ]:
missing = []
corrupted = []
for split_name, df in splits.items():
    print(f'Checking {split_name}: {len(df):,} images...')
    for filename in df['filename']:
        image_path = IMAGE_DIR / filename
        if not image_path.is_file():
            missing.append((split_name, filename))
            continue
        try:
            with Image.open(image_path) as im:
                im.verify()
        except Exception as exc:
            corrupted.append((split_name, filename, str(exc)))

print('Missing images:', len(missing))
print('Corrupted images:', len(corrupted))
pd.DataFrame(missing, columns=['split', 'filename']).to_csv(OUTPUT_DIR / 'missing_images.csv', index=False)
pd.DataFrame(corrupted, columns=['split', 'filename', 'error']).to_csv(OUTPUT_DIR / 'corrupted_images.csv', index=False)

## 8. Verify against documented Week 2 findings

These values are **previously recorded project findings**, not new results. The checks below compare them with the current run; an assertion failure should be investigated rather than ignored.

In [ ]:
EXPECTED = {
    'train': 45095, 'val': 7508, 'test': 22619,
    'total': 75222, 'classes': 102,
    'largest_class': 101, 'largest_count': 3444,
    'smallest_class': 72, 'smallest_count': 42,
    'tail_images': 1864,
    'tail_classes': [72,80,61,63,98,81,36,35,65,17,85,12,75,60,53,55,89,43,96,90,79]
}
for name in ('train','val','test'):
    assert len(splits[name]) == EXPECTED[name], f'{name} count mismatch'
assert sum(len(df) for df in splits.values()) == EXPECTED['total']
assert len(train_counts) == EXPECTED['classes']
assert int(train_counts.idxmax()) == EXPECTED['largest_class']
assert int(train_counts.max()) == EXPECTED['largest_count']
assert int(train_counts.idxmin()) == EXPECTED['smallest_class']
assert int(train_counts.min()) == EXPECTED['smallest_count']
assert int(tail_counts.sum()) == EXPECTED['tail_images']
assert TAIL_CLASSES == EXPECTED['tail_classes'], 'Tail class IDs/order differ from documented results'
assert len(missing) == 0 and len(corrupted) == 0, 'Image integrity check failed'
print('All documented Week 2 findings reproduced successfully.')

## 9. Export analysis summary

In [ ]:
report = {
    'split_sizes': {name: len(df) for name, df in splits.items()},
    'total_images': sum(len(df) for df in splits.values()),
    'num_classes': NUM_CLASSES,
    'largest_class_id': int(train_counts.idxmax()),
    'largest_class_count': int(train_counts.max()),
    'smallest_class_id': int(train_counts.idxmin()),
    'smallest_class_count': int(train_counts.min()),
    'tail_class_ids': TAIL_CLASSES,
    'tail_training_images': int(tail_counts.sum()),
    'missing_images': len(missing),
    'corrupted_images': len(corrupted)
}
with open(OUTPUT_DIR / 'week02_summary.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)
print('Saved analysis files to:', OUTPUT_DIR)
print(json.dumps(report, indent=2))

## Notes on provenance

This notebook was reconstructed after the original Colab code cells were not preserved. It records the documented analysis procedure and expected findings. **Its code cells have not been re-executed as part of notebook generation.** To verify reproducibility, run all cells in a fresh environment with the IP102 dataset.